# 04: MLflow, A/B Testing & Drift Detection

**Module 3 — Week 13: MLOps & Production Systems**

---

## Learning Objectives

1. Understand why experiment tracking is essential and how MLflow organizes ML workflows
2. Track experiments, parameters, metrics, and artifacts with MLflow Tracking
3. Query, compare, and select the best models from recorded runs
4. Use the MLflow Model Registry to manage model lifecycle (Staging, Production, Archived)
5. Design and simulate A/B tests for ML models in production
6. Apply statistical tests (chi-squared, confidence intervals, effect size) to evaluate A/B results
7. Visualize A/B test outcomes with publication-quality figures
8. Detect data drift using PSI and the Kolmogorov-Smirnov test
9. Use the Evidently library for automated drift monitoring
10. Combine tracking, testing, and monitoring into a continuous ML improvement loop

## Resources

- [MLflow Documentation](https://mlflow.org/docs/latest/index.html)
- [MLflow Tracking Quickstart](https://mlflow.org/docs/latest/getting-started/intro-quickstart/index.html)
- [Evidently AI Documentation](https://docs.evidentlyai.com/)
- [Trustworthy Online Controlled Experiments (Kohavi et al.)](https://www.cambridge.org/core/books/trustworthy-online-controlled-experiments/D97B26382EB0EB2DC2019A7A7B518F59)
- [Population Stability Index (PSI) Explained](https://scholarworks.wmich.edu/dissertations/3208/)
- [Practical Statistics for Data Scientists (Bruce et al.)](https://www.oreilly.com/library/view/practical-statistics-for/9781492072935/)

In [ ]:
import sys
sys.path.insert(0, '../../..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report
)
from scipy import stats
from scipy.stats import chi2_contingency, norm
import warnings
import json
import os
import tempfile
from datetime import datetime, timedelta

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Optional imports — MLflow and Evidently
try:
    import mlflow
    import mlflow.sklearn
    MLFLOW_AVAILABLE = True
    print(f"MLflow version: {mlflow.__version__}")
except ImportError:
    MLFLOW_AVAILABLE = False
    print("MLflow not installed. Install with: pip install mlflow")
    print("Proceeding with conceptual examples and manual tracking.")

try:
    import evidently
    from evidently.report import Report
    from evidently.metric_preset import DataDriftPreset, TargetDriftPreset
    EVIDENTLY_AVAILABLE = True
    print(f"Evidently version: {evidently.__version__}")
except ImportError:
    EVIDENTLY_AVAILABLE = False
    print("Evidently not installed. Install with: pip install evidently")
    print("We will use manual PSI-based drift detection as a fallback.")

np.random.seed(42)
print("\nAll core imports loaded successfully.")

---

## Section 1: Why Track Experiments

### The Chaos Without Tracking

Every data scientist has been here:

- "Which model got 0.89 AUC? Was it the one with `max_depth=5` or `max_depth=8`?"
- "I know I tried L2 regularization last week, but what was the C value?"
- "The model in production — which notebook version produced it?"
- "My colleague's model is better than mine. What did they do differently?"

Without systematic tracking, ML development degenerates into:

| Problem | Consequence |
|---------|-------------|
| No parameter logging | Cannot reproduce results |
| No metric history | Cannot compare fairly across runs |
| No artifact versioning | Cannot roll back to a previous model |
| Manual spreadsheets | Error-prone, incomplete, out of date |
| Naming files `model_v2_final_FINAL.pkl` | Self-explanatory chaos |

### MLflow Components

MLflow solves this with four integrated components:

1. **MLflow Tracking** — Record parameters, metrics, and artifacts for every experiment run.
2. **MLflow Projects** — Package ML code in a reusable, reproducible format.
3. **MLflow Models** — A standard format for packaging models for deployment.
4. **MLflow Model Registry** — A centralized store to manage model lifecycle stages.

### Comparison: Experiment Tracking Tools

| Feature | MLflow | Weights & Biases | Neptune | Manual Spreadsheet |
|---------|--------|-------------------|---------|--------------------|
| **Cost** | Free / OSS | Free tier + paid | Free tier + paid | Free |
| **Self-hosted** | Yes | No (SaaS) | No (SaaS) | N/A |
| **Auto-logging** | Yes | Yes | Yes | No |
| **Model Registry** | Yes | Yes | Yes | No |
| **Team collaboration** | Via server | Built-in | Built-in | Manual sharing |
| **Artifact storage** | Local / S3 / GCS | Cloud | Cloud | Manual files |
| **Reproducibility** | Projects format | Artifacts | Artifacts | None |
| **Learning curve** | Low | Low | Medium | None |
| **Scales to prod** | Yes | Yes | Yes | No |

**Our choice for this notebook:** MLflow. It is open-source, runs entirely locally (no account needed), and covers the full lifecycle.

In [ ]:
# --- Section 1: Set up MLflow with a local file store ---

MLFLOW_TRACKING_DIR = os.path.join(os.getcwd(), "mlruns")

if MLFLOW_AVAILABLE:
    # Use a local file-based tracking URI — no server needed
    mlflow.set_tracking_uri(f"file://{MLFLOW_TRACKING_DIR}")
    print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")

    # Create an experiment for our churn modelling work
    experiment_name = "churn-model-comparison"
    experiment = mlflow.set_experiment(experiment_name)
    print(f"Experiment name: {experiment.name}")
    print(f"Experiment ID:   {experiment.experiment_id}")
    print(f"Artifact location: {experiment.artifact_location}")
else:
    print("MLflow not available. We will simulate tracking with dictionaries.")
    # Fallback: manual tracking
    manual_runs = []
    print("Manual run tracker initialized.")

---

## Section 2: MLflow Experiment Tracking

### Core Tracking API

MLflow Tracking is organized around **runs**. Each run records:

- **Parameters** (`log_param`) — Hyperparameters, data paths, feature flags.
- **Metrics** (`log_metric`) — Numeric values like accuracy, loss, latency.
- **Artifacts** (`log_artifact`) — Files such as plots, serialized models, data samples.
- **Tags** (`set_tag`) — Metadata like author name, run purpose, git commit hash.

The recommended pattern is the **context manager**:

```python
with mlflow.start_run(run_name="my-run"):
    mlflow.log_param("learning_rate", 0.01)
    mlflow.log_metric("accuracy", 0.92)
    mlflow.log_artifact("confusion_matrix.png")
```

This ensures the run is always properly closed, even if an exception occurs.

### Autologging

For scikit-learn, MLflow can **automatically** log parameters, metrics, and the model:

```python
mlflow.sklearn.autolog()
```

We will use **explicit logging** below so you see every step, then demonstrate autologging as a comparison.

In [ ]:
# --- Section 2: Generate synthetic churn dataset ---

def generate_churn_data(n_samples=5000, random_state=42):
    """
    Generate a synthetic customer churn dataset.

    Features:
      - tenure: months as a customer (0-72)
      - monthly_charges: monthly bill amount
      - total_charges: cumulative charges
      - contract_type: 0=month-to-month, 1=one-year, 2=two-year
      - num_support_tickets: support interactions
      - has_online_security: binary
      - has_tech_support: binary
      - avg_monthly_usage_gb: data usage in GB

    Target: churned (1) or retained (0)
    """
    rng = np.random.RandomState(random_state)

    tenure = rng.uniform(0, 72, n_samples)
    monthly_charges = rng.uniform(20, 120, n_samples)
    total_charges = tenure * monthly_charges + rng.normal(0, 50, n_samples)
    total_charges = np.clip(total_charges, 0, None)
    contract_type = rng.choice([0, 1, 2], n_samples, p=[0.5, 0.3, 0.2])
    num_support_tickets = rng.poisson(2, n_samples)
    has_online_security = rng.binomial(1, 0.4, n_samples)
    has_tech_support = rng.binomial(1, 0.35, n_samples)
    avg_monthly_usage_gb = rng.exponential(50, n_samples)

    # Churn probability: higher for short tenure, high charges,
    # month-to-month contracts, many support tickets
    churn_logit = (
        -0.03 * tenure
        + 0.015 * monthly_charges
        + 0.8 * (contract_type == 0).astype(float)
        + 0.15 * num_support_tickets
        - 0.5 * has_online_security
        - 0.4 * has_tech_support
        - 0.005 * avg_monthly_usage_gb
        + rng.normal(0, 0.5, n_samples)
    )
    churn_prob = 1 / (1 + np.exp(-churn_logit))
    churned = rng.binomial(1, churn_prob)

    df = pd.DataFrame({
        'tenure': tenure,
        'monthly_charges': monthly_charges,
        'total_charges': total_charges,
        'contract_type': contract_type,
        'num_support_tickets': num_support_tickets,
        'has_online_security': has_online_security,
        'has_tech_support': has_tech_support,
        'avg_monthly_usage_gb': avg_monthly_usage_gb,
        'churned': churned
    })
    return df


churn_df = generate_churn_data(n_samples=5000)
print(f"Dataset shape: {churn_df.shape}")
print(f"Churn rate: {churn_df['churned'].mean():.2%}")
print()
churn_df.head()

In [ ]:
# --- Section 2: Train 3 models with MLflow tracking ---

# Prepare data
feature_cols = [
    'tenure', 'monthly_charges', 'total_charges', 'contract_type',
    'num_support_tickets', 'has_online_security', 'has_tech_support',
    'avg_monthly_usage_gb'
]
X = churn_df[feature_cols]
y = churn_df['churned']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train_scaled.shape[0]} samples")
print(f"Test set:     {X_test_scaled.shape[0]} samples")


# Define models to compare
models = {
    'LogisticRegression': LogisticRegression(
        C=1.0, max_iter=1000, random_state=42
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=100, max_depth=10, random_state=42
    ),
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42
    ),
}


def save_confusion_matrix(y_true, y_pred, model_name, filepath):
    """Save a confusion matrix plot as an artifact."""
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Retained', 'Churned'],
                yticklabels=['Retained', 'Churned'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'Confusion Matrix — {model_name}')
    fig.tight_layout()
    fig.savefig(filepath, dpi=100)
    plt.close(fig)


# Store results for later comparison (works with or without MLflow)
run_results = []

for model_name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Training: {model_name}")
    print(f"{'='*50}")

    if MLFLOW_AVAILABLE:
        with mlflow.start_run(run_name=model_name):
            # Log parameters
            mlflow.set_tag("model_type", model_name)
            mlflow.set_tag("dataset", "synthetic_churn")
            mlflow.set_tag("author", "week13-notebook")

            params = model.get_params()
            for param_name, param_value in params.items():
                mlflow.log_param(param_name, param_value)
            mlflow.log_param("n_features", X_train_scaled.shape[1])
            mlflow.log_param("n_train_samples", X_train_scaled.shape[0])

            # Train
            model.fit(X_train_scaled, y_train)

            # Predict
            y_pred = model.predict(X_test_scaled)
            y_proba = model.predict_proba(X_test_scaled)[:, 1]

            # Calculate metrics
            acc = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred)
            auc = roc_auc_score(y_test, y_proba)

            # Log metrics
            mlflow.log_metric("accuracy", acc)
            mlflow.log_metric("f1_score", f1)
            mlflow.log_metric("auc", auc)

            # Log confusion matrix as artifact
            cm_path = os.path.join(tempfile.gettempdir(), f"cm_{model_name}.png")
            save_confusion_matrix(y_test, y_pred, model_name, cm_path)
            mlflow.log_artifact(cm_path, artifact_path="plots")

            # Log the model itself
            mlflow.sklearn.log_model(model, artifact_path="model")

            run_id = mlflow.active_run().info.run_id
            print(f"  Run ID:   {run_id}")
            print(f"  Accuracy: {acc:.4f}")
            print(f"  F1 Score: {f1:.4f}")
            print(f"  AUC:      {auc:.4f}")

            run_results.append({
                'run_id': run_id,
                'model_name': model_name,
                'accuracy': acc,
                'f1_score': f1,
                'auc': auc,
                'model_obj': model
            })
    else:
        # Fallback: train without MLflow
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]

        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_proba)

        print(f"  Accuracy: {acc:.4f}")
        print(f"  F1 Score: {f1:.4f}")
        print(f"  AUC:      {auc:.4f}")

        run_results.append({
            'run_id': f'manual-{model_name}',
            'model_name': model_name,
            'accuracy': acc,
            'f1_score': f1,
            'auc': auc,
            'model_obj': model
        })

print("\nAll models trained and logged.")

---

## Section 3: Querying and Comparing Runs

### Why Querying Matters

After running many experiments, you need to answer questions programmatically:

- "Show me all runs where AUC > 0.80."
- "Which model had the highest F1 score?"
- "Compare all RandomForest runs sorted by accuracy."

MLflow's `search_runs()` returns a pandas DataFrame, making analysis seamless.

```python
# Example filter syntax
runs = mlflow.search_runs(
    filter_string="metrics.auc > 0.80",
    order_by=["metrics.auc DESC"]
)
```

In [ ]:
# --- Section 3: Query all runs and display as DataFrame ---

if MLFLOW_AVAILABLE:
    # Fetch all runs from our experiment
    runs_df = mlflow.search_runs(
        experiment_names=["churn-model-comparison"],
        order_by=["metrics.auc DESC"]
    )

    # Select key columns for display
    display_cols = [
        'run_id', 'tags.model_type', 'status',
        'metrics.accuracy', 'metrics.f1_score', 'metrics.auc'
    ]
    available_cols = [c for c in display_cols if c in runs_df.columns]
    print("All experiment runs (sorted by AUC descending):")
    print(runs_df[available_cols].to_string(index=False))

    # Find the best model by AUC
    best_run = runs_df.iloc[0]
    print(f"\nBest model by AUC:")
    print(f"  Model:    {best_run.get('tags.model_type', 'N/A')}")
    print(f"  Run ID:   {best_run['run_id']}")
    print(f"  AUC:      {best_run['metrics.auc']:.4f}")
    print(f"  F1:       {best_run['metrics.f1_score']:.4f}")
    print(f"  Accuracy: {best_run['metrics.accuracy']:.4f}")

    best_run_id = best_run['run_id']
else:
    # Fallback: use our manual results
    results_df = pd.DataFrame(run_results).drop(columns=['model_obj'])
    results_df = results_df.sort_values('auc', ascending=False).reset_index(drop=True)
    print("All experiment runs (sorted by AUC descending):")
    print(results_df.to_string(index=False))

    best_row = results_df.iloc[0]
    best_run_id = best_row['run_id']
    print(f"\nBest model by AUC: {best_row['model_name']} (AUC={best_row['auc']:.4f})")

In [ ]:
# --- Section 3: Plot metric comparison across models ---

results_for_plot = pd.DataFrame([
    {k: v for k, v in r.items() if k != 'model_obj'}
    for r in run_results
])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics = ['accuracy', 'f1_score', 'auc']
titles = ['Accuracy', 'F1 Score', 'AUC-ROC']
colors = ['#2196F3', '#4CAF50', '#FF9800']

for ax, metric, title, color in zip(axes, metrics, titles, colors):
    bars = ax.bar(
        results_for_plot['model_name'], results_for_plot[metric],
        color=color, alpha=0.8, edgecolor='black', linewidth=0.5
    )
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_ylabel(title)
    ax.set_ylim(0.5, 1.0)

    # Add value labels on bars
    for bar, val in zip(bars, results_for_plot[metric]):
        ax.text(
            bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold'
        )
    ax.tick_params(axis='x', rotation=20)

fig.suptitle('Model Comparison — Churn Prediction', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## Section 4: MLflow Model Registry

### From Runs to Registry

The **Model Registry** is a centralized model store that enables:

- **Versioning** — Every registered model gets an auto-incrementing version number.
- **Lifecycle Stages** — Models transition through well-defined stages:

```
None  -->  Staging  -->  Production  -->  Archived
                \                          /
                 +------------------------+
                 (can go back to Staging)
```

- **Annotations** — Comments and descriptions at each transition.
- **CI/CD Integration** — Promotion can be gated by automated tests.

### Typical Workflow

1. Train and log a model with `mlflow.sklearn.log_model()`.
2. Register it: `mlflow.register_model(model_uri, name)`.
3. Transition to Staging: run validation tests.
4. Promote to Production: after A/B test or stakeholder approval.
5. When a newer model is promoted, the old one is Archived.

In [ ]:
# --- Section 4: Model Registry operations ---

REGISTRY_MODEL_NAME = "churn-prediction-model"

if MLFLOW_AVAILABLE:
    # Register the best model
    model_uri = f"runs:/{best_run_id}/model"
    print(f"Registering model from: {model_uri}")

    try:
        model_version = mlflow.register_model(
            model_uri=model_uri,
            name=REGISTRY_MODEL_NAME
        )
        print(f"Registered model: {model_version.name}")
        print(f"Version:          {model_version.version}")
        print(f"Stage:            {model_version.current_stage}")
    except Exception as e:
        print(f"Registration note: {e}")
        print("This is expected if the model was already registered.")

    # Transition to Staging
    client = mlflow.MlflowClient()
    try:
        client.transition_model_version_stage(
            name=REGISTRY_MODEL_NAME,
            version=1,
            stage="Staging",
            archive_existing_versions=False
        )
        print(f"\nModel transitioned to: Staging")
    except Exception as e:
        print(f"Staging transition note: {e}")

    # Transition to Production
    try:
        client.transition_model_version_stage(
            name=REGISTRY_MODEL_NAME,
            version=1,
            stage="Production",
            archive_existing_versions=False
        )
        print(f"Model transitioned to: Production")
    except Exception as e:
        print(f"Production transition note: {e}")

    # Load the model back from the registry
    try:
        prod_model_uri = f"models:/{REGISTRY_MODEL_NAME}/Production"
        loaded_model = mlflow.sklearn.load_model(prod_model_uri)
        sample_pred = loaded_model.predict(X_test_scaled[:5])
        print(f"\nLoaded production model predictions (first 5): {sample_pred}")
    except Exception as e:
        print(f"\nLoad note: {e}")
        print("Loading by stage name requires MLflow >= 2.0 with aliases.")
        # Alternative: load by run_id
        loaded_model = mlflow.sklearn.load_model(f"runs:/{best_run_id}/model")
        sample_pred = loaded_model.predict(X_test_scaled[:5])
        print(f"Loaded model via run_id predictions (first 5): {sample_pred}")

else:
    print("MLflow not available. Demonstrating the concept:")
    print()
    print("Registry workflow:")
    print("  1. mlflow.register_model('runs:/<run_id>/model', 'churn-prediction-model')")
    print("  2. client.transition_model_version_stage(name, version=1, stage='Staging')")
    print("  3. client.transition_model_version_stage(name, version=1, stage='Production')")
    print("  4. model = mlflow.sklearn.load_model('models:/churn-prediction-model/Production')")
    print()
    # Use the best model from our manual tracking
    best_model_name = results_for_plot.sort_values('auc', ascending=False).iloc[0]['model_name']
    loaded_model = [r['model_obj'] for r in run_results if r['model_name'] == best_model_name][0]
    sample_pred = loaded_model.predict(X_test_scaled[:5])
    print(f"Best model ({best_model_name}) predictions (first 5): {sample_pred}")

---

## Section 5: A/B Testing — Design and Simulation

### Why A/B Test ML Models?

Offline metrics (AUC, F1) tell you how well a model **fits historical data**. They do **not** tell you:

- How users will react to the model's recommendations.
- Whether the model improves the **business metric** you care about (revenue, retention, engagement).
- Whether the model behaves well on the latest distribution of data.

**A/B testing** is the gold standard for measuring **causal impact** in production:

1. **Randomize** users into two groups: Control (Model A, the current production model) and Treatment (Model B, the challenger).
2. **Observe** a business metric (e.g., churn rate after intervention).
3. **Analyze** whether the difference is statistically significant.

### Key Design Decisions

| Decision | Considerations |
|----------|---------------|
| **Randomization unit** | User-level (most common), session-level, or request-level |
| **Sample size** | Must be large enough to detect the expected effect |
| **Duration** | Long enough to capture weekly patterns (usually 1-4 weeks) |
| **Traffic split** | 50/50 for fastest results; smaller treatment for safety |
| **Metric** | Primary metric (e.g., churn) plus guardrail metrics (e.g., latency) |
| **Stopping rule** | Pre-registered: run for N days, do NOT peek and stop early |

In [ ]:
# --- Section 5: A/B Test Simulator ---

class ABTestSimulator:
    """
    Simulate an A/B test comparing two ML models that predict churn
    and trigger a retention intervention.

    Model A (control): The current production model.
    Model B (treatment): A new challenger model.

    The business metric is the **post-intervention churn rate**.
    A better model identifies at-risk customers more accurately,
    leading to better-targeted interventions and lower churn.
    """

    def __init__(self, random_state=42):
        self.rng = np.random.RandomState(random_state)

    def simulate(
        self,
        n_users=10000,
        model_a_churn_rate=0.15,
        model_b_churn_rate=0.12,
        traffic_split=0.5,
        cost_per_intervention=10.0,
        revenue_per_retained=100.0,
        duration_days=28
    ):
        """
        Simulate A/B test results.

        Parameters
        ----------
        n_users : int
            Total number of users in the experiment.
        model_a_churn_rate : float
            Expected churn rate under Model A (control).
        model_b_churn_rate : float
            Expected churn rate under Model B (treatment).
        traffic_split : float
            Fraction of traffic assigned to Model B.
        cost_per_intervention : float
            Cost of each retention intervention.
        revenue_per_retained : float
            Revenue saved per retained customer.
        duration_days : int
            Duration of the experiment.

        Returns
        -------
        pd.DataFrame
            One row per user with group assignment and outcome.
        """
        # Assign users to groups
        n_b = int(n_users * traffic_split)
        n_a = n_users - n_b

        # Generate outcomes
        churned_a = self.rng.binomial(1, model_a_churn_rate, n_a)
        churned_b = self.rng.binomial(1, model_b_churn_rate, n_b)

        # Simulate enrollment dates spread across the duration
        days_a = self.rng.randint(0, duration_days, n_a)
        days_b = self.rng.randint(0, duration_days, n_b)

        df_a = pd.DataFrame({
            'user_id': range(0, n_a),
            'group': 'A',
            'model': 'control',
            'churned': churned_a,
            'enrollment_day': days_a,
            'intervention_cost': cost_per_intervention,
            'revenue_if_retained': revenue_per_retained
        })

        df_b = pd.DataFrame({
            'user_id': range(n_a, n_a + n_b),
            'group': 'B',
            'model': 'treatment',
            'churned': churned_b,
            'enrollment_day': days_b,
            'intervention_cost': cost_per_intervention,
            'revenue_if_retained': revenue_per_retained
        })

        results = pd.concat([df_a, df_b], ignore_index=True)

        # Shuffle to remove ordering bias
        results = results.sample(frac=1, random_state=self.rng).reset_index(drop=True)

        return results

    def analyze(self, results_df):
        """
        Compute summary statistics for an A/B test.

        Returns a dictionary with churn rates, lift, cost analysis,
        and sample sizes per group.
        """
        group_stats = results_df.groupby('group').agg(
            n_users=('user_id', 'count'),
            n_churned=('churned', 'sum'),
            churn_rate=('churned', 'mean'),
            total_intervention_cost=('intervention_cost', 'sum'),
        ).reset_index()

        a_stats = group_stats[group_stats['group'] == 'A'].iloc[0]
        b_stats = group_stats[group_stats['group'] == 'B'].iloc[0]

        absolute_diff = b_stats['churn_rate'] - a_stats['churn_rate']
        relative_lift = absolute_diff / a_stats['churn_rate']

        # Revenue impact
        rev_per_user = results_df['revenue_if_retained'].iloc[0]
        additional_retained = (a_stats['churn_rate'] - b_stats['churn_rate']) * b_stats['n_users']
        additional_revenue = additional_retained * rev_per_user

        analysis = {
            'group_stats': group_stats,
            'churn_rate_a': a_stats['churn_rate'],
            'churn_rate_b': b_stats['churn_rate'],
            'absolute_diff': absolute_diff,
            'relative_lift': relative_lift,
            'additional_retained_users': additional_retained,
            'additional_revenue': additional_revenue,
            'n_users_a': int(a_stats['n_users']),
            'n_users_b': int(b_stats['n_users']),
        }
        return analysis


# Run the simulation
simulator = ABTestSimulator(random_state=42)

ab_results = simulator.simulate(
    n_users=10000,
    model_a_churn_rate=0.15,   # Current model: 15% churn
    model_b_churn_rate=0.12,   # New model: 12% churn (20% reduction)
    traffic_split=0.5,
    cost_per_intervention=10.0,
    revenue_per_retained=100.0,
    duration_days=28
)

print(f"Simulation complete: {len(ab_results)} users")
print(f"Group A (control):   {(ab_results['group'] == 'A').sum()} users")
print(f"Group B (treatment): {(ab_results['group'] == 'B').sum()} users")
print()

# Summary analysis
analysis = simulator.analyze(ab_results)

print("A/B Test Summary")
print("=" * 45)
print(f"Churn rate (A — control):   {analysis['churn_rate_a']:.4f}")
print(f"Churn rate (B — treatment): {analysis['churn_rate_b']:.4f}")
print(f"Absolute difference:        {analysis['absolute_diff']:.4f}")
print(f"Relative lift:              {analysis['relative_lift']:.2%}")
print(f"Additional retained users:  {analysis['additional_retained_users']:.0f}")
print(f"Additional revenue:         ${analysis['additional_revenue']:,.0f}")

---

## Section 6: Statistical Tests for A/B

### Choosing the Right Test

Our outcome is binary (churned vs. not churned), so we use tests for **proportions**:

1. **Chi-squared test** — Tests whether the churn proportions differ between groups.
2. **Z-test for proportions** — Equivalent; gives a confidence interval.
3. **Fisher's exact test** — For small samples (not our case here).

### Key Statistical Concepts

| Concept | Meaning | Our Context |
|---------|---------|-------------|
| **p-value** | Probability of observing this result (or more extreme) under H0 | H0: churn rates are equal |
| **Confidence interval** | Range of plausible values for the true difference | 95% CI for churn rate difference |
| **Effect size (Cohen's h)** | Standardized measure of the difference | 0.2 = small, 0.5 = medium, 0.8 = large |
| **Power** | Probability of detecting a real effect | We want power >= 0.80 |
| **Type I error (alpha)** | False positive rate | Typically 0.05 |
| **Type II error (beta)** | False negative rate | 1 - power |

### What p-values Do NOT Mean

- A p-value is **not** the probability that H0 is true.
- A p-value < 0.05 does **not** mean the effect is large or important.
- A p-value > 0.05 does **not** mean there is no effect (you may be underpowered).
- Statistical significance is **not** the same as practical significance.

In [ ]:
# --- Section 6: Full statistical analysis of the A/B test ---

def ab_test_statistical_analysis(results_df, alpha=0.05):
    """
    Perform a comprehensive statistical analysis of A/B test results.

    Returns a dictionary with test statistics, p-values,
    confidence intervals, effect size, and power.
    """
    a_data = results_df[results_df['group'] == 'A']['churned']
    b_data = results_df[results_df['group'] == 'B']['churned']

    n_a, n_b = len(a_data), len(b_data)
    p_a, p_b = a_data.mean(), b_data.mean()
    x_a, x_b = a_data.sum(), b_data.sum()

    # --- 1. Chi-squared test ---
    contingency = np.array([
        [x_a, n_a - x_a],   # Group A: churned, not churned
        [x_b, n_b - x_b],   # Group B: churned, not churned
    ])
    chi2, chi2_p, dof, expected = chi2_contingency(contingency, correction=False)

    # --- 2. Z-test for difference in proportions ---
    p_pooled = (x_a + x_b) / (n_a + n_b)
    se_pooled = np.sqrt(p_pooled * (1 - p_pooled) * (1/n_a + 1/n_b))
    z_stat = (p_a - p_b) / se_pooled
    z_p = 2 * (1 - norm.cdf(abs(z_stat)))  # Two-tailed

    # --- 3. Confidence interval for the difference ---
    se_diff = np.sqrt(p_a * (1 - p_a) / n_a + p_b * (1 - p_b) / n_b)
    z_crit = norm.ppf(1 - alpha / 2)
    diff = p_a - p_b  # Positive means B is better (lower churn)
    ci_lower = diff - z_crit * se_diff
    ci_upper = diff + z_crit * se_diff

    # --- 4. Effect size: Cohen's h ---
    # h = 2 * arcsin(sqrt(p1)) - 2 * arcsin(sqrt(p2))
    cohens_h = 2 * np.arcsin(np.sqrt(p_a)) - 2 * np.arcsin(np.sqrt(p_b))

    # --- 5. Post-hoc power analysis ---
    # Power = P(reject H0 | H1 is true)
    noncentrality = abs(p_a - p_b) / se_pooled
    power = 1 - norm.cdf(z_crit - noncentrality) + norm.cdf(-z_crit - noncentrality)

    # --- 6. Sample size needed (for future reference) ---
    # n = (z_alpha/2 + z_beta)^2 * (p1(1-p1) + p2(1-p2)) / (p1 - p2)^2
    z_beta = norm.ppf(0.80)  # 80% power
    n_required = (
        (z_crit + z_beta) ** 2
        * (p_a * (1 - p_a) + p_b * (1 - p_b))
        / (p_a - p_b) ** 2
    )

    return {
        'chi2_statistic': chi2,
        'chi2_p_value': chi2_p,
        'z_statistic': z_stat,
        'z_p_value': z_p,
        'p_a': p_a,
        'p_b': p_b,
        'difference': diff,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'cohens_h': cohens_h,
        'power': power,
        'n_per_group_required': int(np.ceil(n_required)),
        'significant': z_p < alpha,
        'alpha': alpha,
    }


# Run the analysis
stats_result = ab_test_statistical_analysis(ab_results, alpha=0.05)

print("Statistical Analysis of A/B Test")
print("=" * 55)
print(f"")
print(f"Chi-squared statistic:  {stats_result['chi2_statistic']:.4f}")
print(f"Chi-squared p-value:    {stats_result['chi2_p_value']:.6f}")
print(f"")
print(f"Z-statistic:            {stats_result['z_statistic']:.4f}")
print(f"Z-test p-value:         {stats_result['z_p_value']:.6f}")
print(f"")
print(f"Churn rate A (control): {stats_result['p_a']:.4f}")
print(f"Churn rate B (treatment): {stats_result['p_b']:.4f}")
print(f"Difference (A - B):     {stats_result['difference']:.4f}")
print(f"95% CI for difference:  [{stats_result['ci_lower']:.4f}, {stats_result['ci_upper']:.4f}]")
print(f"")
print(f"Cohen's h (effect size): {stats_result['cohens_h']:.4f}")
effect_label = 'small' if abs(stats_result['cohens_h']) < 0.3 else ('medium' if abs(stats_result['cohens_h']) < 0.5 else 'large')
print(f"  Interpretation:        {effect_label}")
print(f"")
print(f"Statistical power:       {stats_result['power']:.4f}")
print(f"Sample size needed per group (80% power): {stats_result['n_per_group_required']:,}")
print(f"")
print(f"Significant at alpha={stats_result['alpha']}? {'YES' if stats_result['significant'] else 'NO'}")

if stats_result['significant']:
    print(f"\nConclusion: Model B significantly reduces churn compared to Model A.")
    print(f"The 95% CI for the improvement is [{stats_result['ci_lower']:.4f}, {stats_result['ci_upper']:.4f}].")
else:
    print(f"\nConclusion: Not enough evidence to conclude Model B is better.")
    print(f"Consider running the test longer or with more users.")

---

## Section 7: Visualizing A/B Results

Good visualizations communicate A/B test results to stakeholders who may not be fluent in statistics. We will build a three-panel figure:

1. **Churn rates by group** with confidence intervals.
2. **Cost/revenue impact** showing the financial case.
3. **Cumulative significance over time** showing when the result stabilized.

In [ ]:
# --- Section 7: Publication-quality 3-panel A/B test visualization ---

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# ---- Panel 1: Churn rates with confidence intervals ----
ax1 = axes[0]

groups = ['A (Control)', 'B (Treatment)']
rates = [stats_result['p_a'], stats_result['p_b']]
# Individual CIs for each proportion
z95 = norm.ppf(0.975)
ci_a = z95 * np.sqrt(stats_result['p_a'] * (1 - stats_result['p_a']) / analysis['n_users_a'])
ci_b = z95 * np.sqrt(stats_result['p_b'] * (1 - stats_result['p_b']) / analysis['n_users_b'])
errors = [ci_a, ci_b]

bar_colors = ['#E53935', '#43A047']
bars = ax1.bar(groups, rates, yerr=errors, capsize=8,
               color=bar_colors, alpha=0.85, edgecolor='black', linewidth=0.5)
ax1.set_ylabel('Churn Rate', fontsize=12)
ax1.set_title('Churn Rate by Group', fontsize=14, fontweight='bold')
ax1.set_ylim(0, max(rates) * 1.4)
for bar, rate in zip(bars, rates):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.008,
             f'{rate:.2%}', ha='center', va='bottom', fontsize=13, fontweight='bold')

# Significance annotation
if stats_result['significant']:
    sig_text = f"p = {stats_result['z_p_value']:.4f} *"
else:
    sig_text = f"p = {stats_result['z_p_value']:.4f} (n.s.)"
ax1.annotate(sig_text, xy=(0.5, 0.92), xycoords='axes fraction',
             ha='center', fontsize=11, fontstyle='italic',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='gray'))

# ---- Panel 2: Cost / revenue impact ----
ax2 = axes[1]

# Calculate financials per 10,000 users
n_scale = 10000
churned_a = stats_result['p_a'] * n_scale
churned_b = stats_result['p_b'] * n_scale
revenue_per_user = 100
cost_per_intervention = 10

revenue_lost_a = churned_a * revenue_per_user
revenue_lost_b = churned_b * revenue_per_user
intervention_cost = n_scale * cost_per_intervention
net_benefit = revenue_lost_a - revenue_lost_b

categories = ['Revenue Lost\n(Model A)', 'Revenue Lost\n(Model B)', 'Net Benefit\nof Model B']
values = [revenue_lost_a, revenue_lost_b, net_benefit]
bar_colors_2 = ['#E53935', '#43A047', '#1E88E5']

bars2 = ax2.bar(categories, values, color=bar_colors_2, alpha=0.85,
                edgecolor='black', linewidth=0.5)
ax2.set_ylabel('USD (per 10,000 users)', fontsize=12)
ax2.set_title('Financial Impact', fontsize=14, fontweight='bold')
for bar, val in zip(bars2, values):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 500,
             f'${val:,.0f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

# ---- Panel 3: Cumulative significance over time ----
ax3 = axes[2]

# Simulate how p-value evolves as more data arrives
sorted_results = ab_results.sort_values('enrollment_day').reset_index(drop=True)
checkpoints = np.arange(200, len(sorted_results) + 1, 200)
p_values_over_time = []
diffs_over_time = []

for n in checkpoints:
    subset = sorted_results.iloc[:n]
    a_sub = subset[subset['group'] == 'A']['churned']
    b_sub = subset[subset['group'] == 'B']['churned']
    if len(a_sub) < 10 or len(b_sub) < 10:
        p_values_over_time.append(1.0)
        diffs_over_time.append(0.0)
        continue

    cont = np.array([
        [a_sub.sum(), len(a_sub) - a_sub.sum()],
        [b_sub.sum(), len(b_sub) - b_sub.sum()]
    ])
    try:
        _, pv, _, _ = chi2_contingency(cont, correction=False)
    except ValueError:
        pv = 1.0
    p_values_over_time.append(pv)
    diffs_over_time.append(a_sub.mean() - b_sub.mean())

ax3.plot(checkpoints, p_values_over_time, 'b-', linewidth=2, label='p-value')
ax3.axhline(y=0.05, color='red', linestyle='--', linewidth=1.5, label='alpha = 0.05')
ax3.fill_between(checkpoints, 0, 0.05, alpha=0.1, color='red')
ax3.set_xlabel('Cumulative Users Observed', fontsize=12)
ax3.set_ylabel('p-value', fontsize=12)
ax3.set_title('Significance Over Time', fontsize=14, fontweight='bold')
ax3.set_ylim(-0.02, 1.0)
ax3.legend(fontsize=11)

# Mark when significance is first reached
sig_indices = [i for i, p in enumerate(p_values_over_time) if p < 0.05]
if sig_indices:
    first_sig = checkpoints[sig_indices[0]]
    ax3.axvline(x=first_sig, color='green', linestyle=':', linewidth=1.5)
    ax3.annotate(f'First significant\nat n={first_sig}',
                 xy=(first_sig, 0.05), xytext=(first_sig + 500, 0.4),
                 arrowprops=dict(arrowstyle='->', color='green'),
                 fontsize=10, color='green')

fig.suptitle('A/B Test Results — Model A vs Model B',
             fontsize=16, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

---

## Section 8: Data Drift Detection

### What Is Drift?

After a model is deployed, the world changes. Drift is the phenomenon where the statistical properties of data shift over time, degrading model performance.

| Type | Definition | Example |
|------|------------|--------|
| **Data drift** (covariate shift) | Input feature distributions change | Average monthly charges increase due to price hikes |
| **Concept drift** | The relationship between features and target changes | Customers now churn for different reasons than before |
| **Prediction drift** | The distribution of model predictions changes | Model suddenly predicts more churners |

### Population Stability Index (PSI)

PSI measures how much a distribution has shifted between a **reference** (training) period and a **current** (production) period.

**Formula:**

$$PSI = \sum_{i=1}^{B} (p_i^{\text{current}} - p_i^{\text{reference}}) \times \ln\left(\frac{p_i^{\text{current}}}{p_i^{\text{reference}}}\right)$$

Where $B$ is the number of bins and $p_i$ is the proportion of observations in bin $i$.

**Interpretation thresholds:**

| PSI Value | Interpretation |
|-----------|---------------|
| < 0.10 | No significant drift |
| 0.10 - 0.25 | Moderate drift — investigate |
| > 0.25 | Significant drift — retrain model |

### Kolmogorov-Smirnov (KS) Test

The KS test is a non-parametric test that compares two distributions by measuring the maximum distance between their cumulative distribution functions. It produces a test statistic and a p-value: if p < 0.05, the distributions are significantly different.

In [ ]:
# --- Section 8: PSI and KS drift detection ---

def compute_psi(reference, current, n_bins=10, epsilon=1e-4):
    """
    Compute the Population Stability Index (PSI) between two distributions.

    Parameters
    ----------
    reference : array-like
        Baseline distribution (e.g., training data).
    current : array-like
        Current distribution (e.g., production data).
    n_bins : int
        Number of quantile bins.
    epsilon : float
        Small value to avoid log(0).

    Returns
    -------
    float
        PSI value.
    """
    # Use reference quantiles to define bins
    breakpoints = np.percentile(reference, np.linspace(0, 100, n_bins + 1))
    breakpoints[0] = -np.inf
    breakpoints[-1] = np.inf

    # Count proportions in each bin
    ref_counts = np.histogram(reference, bins=breakpoints)[0]
    cur_counts = np.histogram(current, bins=breakpoints)[0]

    ref_pct = ref_counts / len(reference) + epsilon
    cur_pct = cur_counts / len(current) + epsilon

    psi = np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct))
    return psi


def detect_drift(reference_df, current_df, features, alpha=0.05):
    """
    Detect drift across multiple features using both PSI and the KS test.

    Parameters
    ----------
    reference_df : pd.DataFrame
        Reference (training/baseline) data.
    current_df : pd.DataFrame
        Current (production) data.
    features : list of str
        Feature columns to check.
    alpha : float
        Significance threshold for KS test.

    Returns
    -------
    pd.DataFrame
        Drift report with PSI, KS statistic, p-value, and drift flag per feature.
    """
    results = []
    for feat in features:
        ref_vals = reference_df[feat].dropna().values
        cur_vals = current_df[feat].dropna().values

        psi = compute_psi(ref_vals, cur_vals)
        ks_stat, ks_p = stats.ks_2samp(ref_vals, cur_vals)

        if psi > 0.25:
            psi_status = 'SIGNIFICANT'
        elif psi > 0.10:
            psi_status = 'MODERATE'
        else:
            psi_status = 'STABLE'

        results.append({
            'feature': feat,
            'psi': psi,
            'psi_status': psi_status,
            'ks_statistic': ks_stat,
            'ks_p_value': ks_p,
            'ks_drift': ks_p < alpha,
        })

    return pd.DataFrame(results)


# --- Simulate gradual drift over time ---

print("Simulating gradual data drift over 6 months...\n")

# Reference data = first 3 months (our training data)
reference_data = churn_df.copy()

# Simulate 6 monthly snapshots with increasing drift
monthly_snapshots = {}
drift_multipliers = [0.0, 0.1, 0.2, 0.4, 0.7, 1.0]  # Increasing drift

for month, drift_level in enumerate(drift_multipliers, start=1):
    snapshot = churn_df.copy()

    # Apply drift to selected features
    # tenure decreases (newer customers)
    snapshot['tenure'] = snapshot['tenure'] - drift_level * 10 + np.random.normal(0, 2, len(snapshot))
    snapshot['tenure'] = np.clip(snapshot['tenure'], 0, 72)

    # monthly_charges increase (price hikes)
    snapshot['monthly_charges'] = snapshot['monthly_charges'] + drift_level * 20

    # more support tickets (service degradation)
    snapshot['num_support_tickets'] = np.clip(
        snapshot['num_support_tickets'] + np.random.poisson(drift_level * 2, len(snapshot)),
        0, None
    )

    # avg_monthly_usage_gb shifts (different user behavior)
    snapshot['avg_monthly_usage_gb'] = snapshot['avg_monthly_usage_gb'] * (1 + drift_level * 0.3)

    monthly_snapshots[f'month_{month}'] = snapshot

# Run drift detection on each monthly snapshot
features_to_monitor = [
    'tenure', 'monthly_charges', 'total_charges', 'num_support_tickets',
    'avg_monthly_usage_gb'
]

print("Per-month drift report (PSI):")
print("=" * 65)

all_drift_reports = []
for month_label, snapshot in monthly_snapshots.items():
    report = detect_drift(reference_data, snapshot, features_to_monitor)
    report['month'] = month_label
    all_drift_reports.append(report)

    # Print summary per month
    n_drifted = (report['psi_status'] != 'STABLE').sum()
    max_psi = report['psi'].max()
    print(f"  {month_label}: {n_drifted}/{len(features_to_monitor)} features drifted (max PSI = {max_psi:.3f})")

# Show detailed report for the last month
print(f"\nDetailed drift report for month_6 (highest drift):")
print(all_drift_reports[-1][['feature', 'psi', 'psi_status', 'ks_statistic', 'ks_p_value', 'ks_drift']].to_string(index=False))

In [ ]:
# --- Section 8: Visualize drift over time ---

# Build a PSI-over-time matrix
psi_matrix = pd.DataFrame()
for report in all_drift_reports:
    month = report['month'].iloc[0]
    for _, row in report.iterrows():
        psi_matrix.loc[month, row['feature']] = row['psi']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: PSI heatmap
ax1 = axes[0]
sns.heatmap(
    psi_matrix.astype(float), annot=True, fmt='.3f', cmap='YlOrRd',
    ax=ax1, linewidths=0.5, vmin=0, vmax=0.5,
    cbar_kws={'label': 'PSI'}
)
ax1.set_title('PSI by Feature Over Time', fontsize=14, fontweight='bold')
ax1.set_ylabel('Month')
ax1.set_xlabel('Feature')
# Draw threshold lines on the colorbar
ax1.axhline(y=0, color='black', linewidth=2)

# Panel 2: PSI line plot per feature
ax2 = axes[1]
months = range(1, 7)
for feat in features_to_monitor:
    psi_vals = [psi_matrix.iloc[i][feat] for i in range(6)]
    ax2.plot(months, psi_vals, 'o-', linewidth=2, markersize=6, label=feat)

ax2.axhline(y=0.10, color='orange', linestyle='--', linewidth=1.5, label='Moderate (0.10)')
ax2.axhline(y=0.25, color='red', linestyle='--', linewidth=1.5, label='Significant (0.25)')
ax2.fill_between(months, 0, 0.10, alpha=0.05, color='green')
ax2.fill_between(months, 0.10, 0.25, alpha=0.05, color='orange')
ax2.fill_between(months, 0.25, ax2.get_ylim()[1] if ax2.get_ylim()[1] > 0.25 else 0.6,
                 alpha=0.05, color='red')
ax2.set_xlabel('Month', fontsize=12)
ax2.set_ylabel('PSI', fontsize=12)
ax2.set_title('Feature Drift Over Time (PSI)', fontsize=14, fontweight='bold')
ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax2.set_xticks(months)

plt.tight_layout()
plt.show()

# Distribution comparison for the most drifted feature
most_drifted_feat = all_drift_reports[-1].sort_values('psi', ascending=False).iloc[0]['feature']
print(f"\nMost drifted feature: {most_drifted_feat}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
months_to_show = [0, 2, 5]  # month 1, 3, 6
titles_show = ['Month 1 (No Drift)', 'Month 3 (Moderate)', 'Month 6 (Severe)']

for ax, m_idx, title in zip(axes, months_to_show, titles_show):
    ref_vals = reference_data[most_drifted_feat]
    cur_vals = list(monthly_snapshots.values())[m_idx][most_drifted_feat]

    ax.hist(ref_vals, bins=40, alpha=0.5, density=True, label='Reference', color='blue')
    ax.hist(cur_vals, bins=40, alpha=0.5, density=True, label='Current', color='red')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel(most_drifted_feat)
    ax.set_ylabel('Density')
    ax.legend()

fig.suptitle(f'Distribution Shift: {most_drifted_feat}', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## Section 9: Evidently Integration

### What Is Evidently?

[Evidently](https://www.evidentlyai.com/) is an open-source Python library for monitoring ML models in production. It generates interactive reports and dashboards for:

- **Data drift** — Statistical tests on every feature.
- **Target drift** — Changes in the prediction or label distribution.
- **Model performance** — Accuracy, precision, recall, calibration over time.
- **Data quality** — Missing values, duplicates, out-of-range values.

Evidently integrates with MLflow: you can log Evidently reports as MLflow artifacts to keep everything in one place.

### Usage Pattern

```python
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset

report = Report(metrics=[DataDriftPreset()])
report.run(reference_data=ref_df, current_data=cur_df)
report.save_html("drift_report.html")
```

In [ ]:
# --- Section 9: Evidently integration (with fallback) ---

reference_subset = reference_data[features_to_monitor].copy()
current_subset = monthly_snapshots['month_6'][features_to_monitor].copy()

if EVIDENTLY_AVAILABLE:
    print("Running Evidently DataDriftPreset report...")

    report = Report(metrics=[DataDriftPreset()])
    report.run(
        reference_data=reference_subset,
        current_data=current_subset
    )

    # Save as HTML
    report_path = os.path.join(tempfile.gettempdir(), "drift_report.html")
    report.save_html(report_path)
    print(f"Report saved to: {report_path}")

    # Extract results as a dict
    report_dict = report.as_dict()
    print(f"\nEvidently report summary:")
    for metric in report_dict.get('metrics', []):
        metric_id = metric.get('metric', 'unknown')
        result = metric.get('result', {})
        if 'drift_share' in result:
            print(f"  Drift share: {result['drift_share']:.2%}")
            print(f"  Number of drifted features: {result.get('number_of_drifted_columns', 'N/A')}")
            print(f"  Number of columns: {result.get('number_of_columns', 'N/A')}")

    # Log the report as an MLflow artifact
    if MLFLOW_AVAILABLE:
        with mlflow.start_run(run_name="drift-detection-month-6"):
            mlflow.log_artifact(report_path, artifact_path="drift_reports")
            mlflow.set_tag("purpose", "drift_monitoring")
            mlflow.set_tag("month", "6")
            print("\nDrift report logged as MLflow artifact.")

else:
    print("Evidently not installed. Using manual PSI-based drift detection.")
    print("(This is the same detect_drift() function from Section 8.)\n")

    manual_report = detect_drift(reference_data, monthly_snapshots['month_6'], features_to_monitor)

    print("Manual Drift Report — Month 6 vs Reference")
    print("=" * 60)
    print(manual_report.to_string(index=False))

    n_drifted = (manual_report['psi_status'] != 'STABLE').sum()
    print(f"\nSummary: {n_drifted}/{len(features_to_monitor)} features show drift.")

    if n_drifted > len(features_to_monitor) / 2:
        print("ALERT: Majority of features have drifted. Model retraining recommended.")
    elif n_drifted > 0:
        print("WARNING: Some features show drift. Monitor closely.")
    else:
        print("All features stable.")

### Exercises

**Exercise 1: Add a New Model to the MLflow Experiment**

Train an `AdaBoostClassifier` (or any other model you choose) on the same churn data and log it to the same MLflow experiment. Compare its AUC, F1, and accuracy against the three models we already trained. Which model is best overall?

In [ ]:
# Exercise 1: Your code here
# Hint:
# from sklearn.ensemble import AdaBoostClassifier
# model = AdaBoostClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
#
# if MLFLOW_AVAILABLE:
#     with mlflow.start_run(run_name='AdaBoost'):
#         model.fit(X_train_scaled, y_train)
#         y_pred = model.predict(X_test_scaled)
#         y_proba = model.predict_proba(X_test_scaled)[:, 1]
#         mlflow.log_metric('accuracy', accuracy_score(y_test, y_pred))
#         mlflow.log_metric('f1_score', f1_score(y_test, y_pred))
#         mlflow.log_metric('auc', roc_auc_score(y_test, y_proba))
#         mlflow.sklearn.log_model(model, 'model')
#
# Then query all runs and compare.
pass

**Exercise 2: Calculate Required Sample Size for a 2% Improvement**

Suppose the current production model has a churn rate of 15%. You want to detect a 2 percentage-point improvement (from 15% to 13%) with 80% power at a 5% significance level. How many users do you need per group?

In [ ]:
# Exercise 2: Your code here
# Hint:
# p1 = 0.15  # control churn rate
# p2 = 0.13  # treatment churn rate
# alpha = 0.05
# power = 0.80
# z_alpha = norm.ppf(1 - alpha / 2)
# z_beta = norm.ppf(power)
# n = (z_alpha + z_beta)**2 * (p1*(1-p1) + p2*(1-p2)) / (p1 - p2)**2
# print(f"Required sample size per group: {int(np.ceil(n)):,}")
pass

**Exercise 3: Build a Drift Monitor for Weekly Snapshots**

Write a function `weekly_drift_monitor(reference_df, weekly_snapshots, features)` that:

1. Takes a reference DataFrame and a list of weekly snapshot DataFrames.
2. For each week, computes the drift report (PSI + KS test for each feature).
3. Returns a summary DataFrame with one row per (week, feature) combination.
4. Prints an alert if any feature crosses the PSI > 0.25 threshold.

Test it on simulated weekly data with gradually increasing drift.

In [ ]:
# Exercise 3: Your code here
# Hint:
# def weekly_drift_monitor(reference_df, weekly_snapshots, features):
#     all_reports = []
#     for week_num, snapshot in enumerate(weekly_snapshots, 1):
#         report = detect_drift(reference_df, snapshot, features)
#         report['week'] = week_num
#         all_reports.append(report)
#         # Check for alerts
#         severe = report[report['psi_status'] == 'SIGNIFICANT']
#         if len(severe) > 0:
#             print(f"ALERT Week {week_num}: {list(severe['feature'])} have significant drift!")
#     return pd.concat(all_reports, ignore_index=True)
pass

---

## Section 10: Key Takeaways

### 1. Track Everything with MLflow from Day One

The cost of setting up experiment tracking is small (a few lines of code). The cost of *not* tracking is enormous — lost reproducibility, wasted re-runs, and no audit trail. Use `mlflow.start_run()` as habitually as you use `git commit`.

### 2. A/B Test Before Full Rollout

Offline metrics (AUC, F1, accuracy) evaluate a model against historical data. They cannot predict how a model will perform in the wild, where user behavior, distribution shifts, and feedback loops come into play. A/B testing is the only reliable way to measure **causal impact** on business metrics.

### 3. Monitor Continuously for Drift

A model that works today may fail tomorrow. Data drift, concept drift, and upstream data quality issues are silent killers. Set up automated monitoring (PSI, KS tests, or Evidently reports) and alert when thresholds are breached. Do not wait for a stakeholder to notice that predictions look wrong.

### 4. Statistical Significance != Practical Significance

A p-value below 0.05 tells you the observed effect is unlikely under the null hypothesis. It does **not** tell you the effect is large enough to matter. Always report:

- **Effect size** (Cohen's h, absolute difference)
- **Confidence interval** (the range of plausible true effects)
- **Business impact** (revenue, cost, user experience)

A statistically significant 0.1% improvement in churn rate may not be worth the engineering cost of deploying a new model.

### 5. Automate Drift Detection

Manual drift checks are forgotten, delayed, or inconsistent. Build a pipeline that:

1. Runs drift detection on a schedule (daily or weekly).
2. Logs results to MLflow or a dashboard.
3. Sends alerts when thresholds are crossed.
4. Optionally triggers retraining when drift is severe.

### Summary of Tools and Concepts

| Topic | Tool/Technique | Key Takeaway |
|-------|---------------|-------------|
| Experiment tracking | MLflow Tracking | Log params, metrics, artifacts for every run |
| Model management | MLflow Registry | Version models, manage lifecycle stages |
| A/B testing design | Randomization, power analysis | Pre-register sample size, duration, primary metric |
| A/B analysis | Chi-squared, Z-test, Cohen's h | Report effect size and CI, not just p-values |
| Data drift | PSI, KS test | PSI > 0.25 means significant drift |
| Drift monitoring | Evidently, custom PSI | Automate checks, alert on breaches |

In [ ]:
# --- Cleanup: remove the local mlruns directory if desired ---
# Uncomment the lines below to clean up after running this notebook.
# import shutil
# if os.path.exists(MLFLOW_TRACKING_DIR):
#     shutil.rmtree(MLFLOW_TRACKING_DIR)
#     print(f"Removed {MLFLOW_TRACKING_DIR}")

print("Notebook complete. All sections covered:")
print("  1. Why Track Experiments")
print("  2. MLflow Experiment Tracking")
print("  3. Querying and Comparing Runs")
print("  4. MLflow Model Registry")
print("  5. A/B Testing - Design and Simulation")
print("  6. Statistical Tests for A/B")
print("  7. Visualizing A/B Results")
print("  8. Data Drift Detection")
print("  9. Evidently Integration")
print("  10. Key Takeaways")